# A Training Update That Counts Correctly

Normalize by prediction tokens, accumulate gradients, clip once, and schedule updates.

## Before You Start

Course 10. Entry course: 1. Budget about 75 minutes for reading, code, and one CPU experiment; optional sweeps take longer.

**Concept prerequisites:** optimizer-loop.

## What You Will Be Able to Do

- Compute masked stable cross entropy
- Aggregate unequal microbatches correctly
- Apply clipping and warm-up at optimizer-step boundaries

## The unit of loss is a predicted token

Logits have shape [B,T,V]; integer targets and a Boolean loss mask have shape [B,T]. At each selected position, negative log likelihood is logsumexp(logits) minus the target logit. Convert numeric logits to floating point before subtraction, retaining integer targets and Boolean masks, then subtract the row maximum before exponentiating. Sum selected losses and divide by the number of selected targets. Padding is not an easy prediction to include in the average. An all-masked batch is an input error, not evidence of perfect loss.

To evaluate a dataset, accumulate loss sums and valid-token counts across batches, then divide once. Averaging batch means gives short batches the same weight as long batches. Perplexity is exp(mean NLL) in natural-log units. It is comparable only under a compatible tokenizer, content boundary, and evaluation protocol.

## Accumulation must reproduce a large batch

Suppose two microbatches contain 2 and 6 valid targets. If their mean losses are 3 and 1, the combined mean is (2*3+6*1)/8=1.5, not 2. Scale each microbatch loss sum by the accumulation-window total token count before backward. Then clip the accumulated global gradient norm and take one optimizer step. Clipping each microbatch separately changes the update.

For gradient vector g and threshold c, preserve its direction and bound its global norm, rather than clipping coordinates independently. Measure the norm after dividing by max(abs(g)) so squaring finite large coordinates does not overflow; zero gradients stay zero. Learning-rate schedules advance on optimizer updates, not on every microbatch. Our schedule uses zero-based update index s: warm-up lr=peak*((s+1)/warmup). Form the bounded ratio before multiplying by peak to avoid an overflowing intermediate. After warm-up, cosine-decay from peak to floor across the remaining updates. The final scheduled update reaches the floor when at least two decay points exist.

## Worked Example: Clip the accumulated vector

The accumulated gradient is [3,4], and the norm limit is 2.

1. **Measure:** The L2 norm is sqrt(9+16)=5.
2. **Scale:** Use scale 2/5=0.4, giving [1.2,1.6]. Its direction is unchanged and its norm is 2.
3. **Update:** With SGD learning rate 0.1, subtract [0.12,0.16] from the parameters. Adam adds moment state; clipping still precedes its step.

A correct loss denominator is part of the gradient, not just the logging display.

## Coding Contract

Implement the public functions below. Private helpers are supplied where the exercise focuses on composition. NumPy/pure Python runs in the browser; the separate experiment uses autograd or a simulator when needed.

```python
# Generated starter; edit the functions in the course editor or notebook.
"""Token-weighted loss and optimizer-step utilities. NumPy only."""
import numpy as np

def loss_totals(logits, targets, mask):
    """Return (NLL sum, valid count) for [B,T,V] logits and [B,T] targets/mask.

Targets are integer IDs in [0,V); mask is Boolean. Require finite logits,
matching nonempty shapes, and at least one selected target; else ValueError.
Use stable logsumexp. Do not modify inputs."""
    raise NotImplementedError('Implement this function, then run the checks.')

def clip_gradient(gradient, max_norm):
    """Return a fresh array with global L2 norm at most positive max_norm.

gradient is any finite float array; reject nonpositive/nonfinite max_norm.
Zero gradients stay zero. Preserve direction; do not clip coordinates.
Scale magnitudes before measuring the norm to avoid overflow/underflow."""
    raise NotImplementedError('Implement this function, then run the checks.')

def learning_rate(step, total, warmup, peak, floor=0.0):
    """Zero-based schedule. Integers: 0<=step<total, 0<=warmup<total.

Require 0<=floor<=peak, finite rates. Warm-up: peak*(step+1)/warmup.
Decay: cosine from peak at step=warmup to floor at step=total-1.
If there is only one decay update, use floor for that update."""
    raise NotImplementedError('Implement this function, then run the checks.')
```

## Check Your Understanding

Batch A: loss sum 12 over 4 tokens. Batch B: loss sum 6 over 6 tokens.

- Combined mean NLL

<details><summary>Worked answers</summary>

- 1.8: (12+6)/(4+6)=1.8. The unweighted average of the two batch means would be 2.

</details>

## Experiment

Run `experiments/scratch_training.py` from the repository, or use the self-contained notebook linked below the editor. Start with the declared CPU defaults. The notebook includes any required local model source; no model weights, datasets, API keys, or paid GPU are needed.

- Compare gradients from one full batch with unequal token-weighted microbatches; report maximum absolute error.
- The experiment trains a tiny causal decoder and reports held-out NLL before and after updates. A short toy run establishes execution, not general language competence.

## Transfer Problems

1. Break accumulation by dividing each microbatch mean equally. Explain when that bug is hidden.
2. Compare clipping thresholds at fixed seed and update count; log unclipped norms too.

For each transfer, submit the changed code, one predicted outcome, actual measurements, one negative control, and a limitation. These open experiments need a defensible result, not a predetermined winning curve.

## AI-Assisted Coding Route

1. Predict the worked example and one edge case without assistance.
2. Ask an assistant for one small function or a test critique, not the completed notebook. Supply the exact shapes and contract.
3. Explain every returned operation and inspect the diff. Reject changes to the grader or the intended objective.
4. Run the public tests, then deliberately break a mechanism named in this lesson. Confirm that a behavioral test fails.
5. Compare with the worked solution only after an attempt. Close it and reimplement the difficult part from memory.
6. Record what you could explain unaided, what the assistant supplied, and which measured result supports your conclusion.

## Completion Evidence

A passing grader establishes this small function contract, not research reproduction or production readiness. Submit a reproducible experiment report with seed, split, shapes, budget, baseline, held-out result, failure case, and limitation. Keep failed seeds. Answers to historical/discussion questions should distinguish fact, inference, and an open empirical claim.

## Primary Source

[LLMs from Scratch: training refinements](https://github.com/rasbt/LLMs-from-scratch/tree/main/appendix-D). The numerical examples, code, and synthetic datasets here are original teaching material; the source is for the underlying mechanism and its research context.



## Runtime

CPU is sufficient. Install the libraries below in a fresh environment. Run the starter and checks after implementing the functions; an untouched starter is expected to fail. Run the reference only when ready to compare. Training experiments are mechanism-scale, not paper benchmark reproductions.


In [ ]:
%pip install -q "numpy>=1.26,<3" "torch>=2.2,<3"


In [ ]:
# Generated starter; edit the functions in the course editor or notebook.
"""Token-weighted loss and optimizer-step utilities. NumPy only."""
import numpy as np

def loss_totals(logits, targets, mask):
    """Return (NLL sum, valid count) for [B,T,V] logits and [B,T] targets/mask.

Targets are integer IDs in [0,V); mask is Boolean. Require finite logits,
matching nonempty shapes, and at least one selected target; else ValueError.
Use stable logsumexp. Do not modify inputs."""
    raise NotImplementedError('Implement this function, then run the checks.')

def clip_gradient(gradient, max_norm):
    """Return a fresh array with global L2 norm at most positive max_norm.

gradient is any finite float array; reject nonpositive/nonfinite max_norm.
Zero gradients stay zero. Preserve direction; do not clip coordinates.
Scale magnitudes before measuring the norm to avoid overflow/underflow."""
    raise NotImplementedError('Implement this function, then run the checks.')

def learning_rate(step, total, warmup, peak, floor=0.0):
    """Zero-based schedule. Integers: 0<=step<total, 0<=warmup<total.

Require 0<=floor<=peak, finite rates. Warm-up: peak*(step+1)/warmup.
Decay: cosine from peak at step=warmup to floor at step=total-1.
If there is only one decay update, use floor for that update."""
    raise NotImplementedError('Implement this function, then run the checks.')


In [ ]:
import numpy as np
rng = np.random.default_rng(708)
for shape in [(2, 3, 5), (1, 4, 2), (3, 1, 7)]:
    z = rng.normal(size=shape) * 9 + 10000
    y = rng.integers(shape[-1], size=shape[:-1])
    m = rng.random(shape[:-1]) > .3
    m.flat[0] = True
    before = z.copy()
    terms = []
    for b, t in zip(*np.where(m)):
        row = z[b, t] - z[b, t].max()
        terms.append(-np.log(np.exp(row[y[b, t]]) / np.exp(row).sum()))
    total, count = loss_totals(z, y, m)
    assert np.isclose(total, sum(terms)) and count == len(terms), 'Score only valid targets'
    assert np.array_equal(z, before)
g = np.array([3., 4.])
assert np.allclose(clip_gradient(g, 2), [1.2, 1.6]), 'Clip global norm, not coordinates'
assert np.array_equal(g, [3, 4])
assert np.allclose(clip_gradient(np.zeros((2, 2)), 1), 0)
assert np.allclose(clip_gradient(g, 10), g)
for magnitude, limit in [(1e200, 1.), (1e308, 1.), (1e-200, 1e-201)]:
    gradient = np.array([[magnitude, -magnitude], [0., 0.]])
    before = gradient.copy()
    clipped = clip_gradient(gradient, limit)
    expected = np.array([[1., -1.], [0., 0.]]) * (limit / np.sqrt(2.))
    assert np.allclose(clipped, expected, rtol=1e-14, atol=0), 'Preserve direction across finite magnitude ranges'
    assert np.array_equal(gradient, before) and not np.shares_memory(clipped, gradient)
large = np.array([1e308, -1e308])
assert np.array_equal(clip_gradient(large, 1.5e308), large), 'Large representable norms below the limit must stay unchanged'
mixed = np.array([1e308, 1e-308])
assert np.allclose(clip_gradient(mixed, 5e307), [5e307, 5e-309], rtol=1e-14, atol=0), 'Retain small coordinates when their clipped values are representable'
empty = np.empty((0, 2))
assert clip_gradient(empty, 1).shape == empty.shape
assert np.allclose([learning_rate(i, 6, 2, 1, .1) for i in range(6)], [.5, 1, 1, .775, .325, .1])
assert learning_rate(0, 1, 0, 1, .2) == .2
assert learning_rate(1, 3, 2, 1e308) == 1e308, 'The final warmup update must reach the finite peak'
for peak in (1.,1e308,np.finfo(float).max):
    rates = np.array([learning_rate(step,5,4,peak) for step in range(4)])
    expected = peak*np.array([.25,.5,.75,1.])
    assert np.isfinite(rates).all() and np.allclose(rates,expected,rtol=1e-14,atol=0.)
for thunk in [lambda: loss_totals(np.zeros((1, 2, 3)), np.zeros((1, 2), int), np.zeros((1, 2), bool)), lambda: clip_gradient(g, 0), lambda: learning_rate(6, 6, 2, 1)]:
    try:
        thunk()
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid inputs must fail explicitly')
print('Training-accounting checks passed.')


## Worked Solution

Read the explanation and docstrings alongside the code. Verify nonmutation, masks, normalization, and numerical conventions against your own implementation.


In [ ]:
"""Token-weighted loss and optimizer-step utilities. NumPy only."""
import numpy as np


def loss_totals(logits, targets, mask):
    """Return (NLL sum, valid count) for [B,T,V] logits and [B,T] targets/mask.

    Targets are integer IDs in [0,V); mask is Boolean. Require finite logits,
    matching nonempty shapes, and at least one selected target; else ValueError.
    Use stable logsumexp. Do not modify inputs.
    """
    z, y, m = np.asarray(logits, float), np.asarray(targets), np.asarray(mask)
    if z.ndim != 3 or z.shape[-1] == 0 or y.shape != z.shape[:-1] or m.shape != y.shape:
        raise ValueError('shape mismatch')
    if y.dtype.kind not in 'iu' or m.dtype.kind != 'b' or not m.any():
        raise ValueError('integer targets and a nonempty Boolean mask required')
    if not np.isfinite(z).all() or (y < 0).any() or (y >= z.shape[-1]).any():
        raise ValueError('invalid logits or targets')
    shifted = z - z.max(axis=-1, keepdims=True)
    nll = np.log(np.exp(shifted).sum(axis=-1)) - np.take_along_axis(shifted, y[..., None], -1)[..., 0]
    return float(nll[m].sum()), int(m.sum())


def clip_gradient(gradient, max_norm):
    """Return a fresh array with global L2 norm at most positive max_norm.

    gradient is any finite float array; reject nonpositive/nonfinite max_norm.
    Zero gradients stay zero. Preserve direction; do not clip coordinates.
    Scale magnitudes before measuring the norm to avoid overflow/underflow.
    """
    g = np.asarray(gradient, float)
    if not np.isfinite(g).all() or not np.isfinite(max_norm) or max_norm <= 0:
        raise ValueError('finite gradient and positive limit required')
    magnitude = np.max(np.abs(g), initial=0.)
    if magnitude == 0:
        return g.copy()
    scaled = g / magnitude
    norm = np.linalg.norm(scaled.ravel())
    if magnitude <= max_norm / norm:
        return g.copy()
    factor = (max_norm / norm) / magnitude
    if factor >= np.finfo(float).tiny:
        return g * factor
    return scaled * (max_norm / norm)


def learning_rate(step, total, warmup, peak, floor=0.):
    """Zero-based schedule. Integers: 0<=step<total, 0<=warmup<total.

    Require 0<=floor<=peak, finite rates. Warm-up: peak*(step+1)/warmup.
    Decay: cosine from peak at step=warmup to floor at step=total-1.
    If there is only one decay update, use floor for that update.
    """
    if any(not isinstance(v, int) or isinstance(v, bool) for v in (step, total, warmup)):
        raise ValueError('integer update indices required')
    if not 0 <= step < total or not 0 <= warmup < total or not 0 <= floor <= peak or not np.isfinite([peak, floor]).all():
        raise ValueError('invalid schedule')
    if step < warmup:
        return peak * ((step + 1) / warmup)
    length = total - warmup
    phase = (step - warmup) / (length - 1) if length > 1 else 1.
    return float(floor + .5 * (peak - floor) * (1 + np.cos(np.pi * phase)))


In [ ]:
import numpy as np
rng = np.random.default_rng(708)
for shape in [(2, 3, 5), (1, 4, 2), (3, 1, 7)]:
    z = rng.normal(size=shape) * 9 + 10000
    y = rng.integers(shape[-1], size=shape[:-1])
    m = rng.random(shape[:-1]) > .3
    m.flat[0] = True
    before = z.copy()
    terms = []
    for b, t in zip(*np.where(m)):
        row = z[b, t] - z[b, t].max()
        terms.append(-np.log(np.exp(row[y[b, t]]) / np.exp(row).sum()))
    total, count = loss_totals(z, y, m)
    assert np.isclose(total, sum(terms)) and count == len(terms), 'Score only valid targets'
    assert np.array_equal(z, before)
g = np.array([3., 4.])
assert np.allclose(clip_gradient(g, 2), [1.2, 1.6]), 'Clip global norm, not coordinates'
assert np.array_equal(g, [3, 4])
assert np.allclose(clip_gradient(np.zeros((2, 2)), 1), 0)
assert np.allclose(clip_gradient(g, 10), g)
for magnitude, limit in [(1e200, 1.), (1e308, 1.), (1e-200, 1e-201)]:
    gradient = np.array([[magnitude, -magnitude], [0., 0.]])
    before = gradient.copy()
    clipped = clip_gradient(gradient, limit)
    expected = np.array([[1., -1.], [0., 0.]]) * (limit / np.sqrt(2.))
    assert np.allclose(clipped, expected, rtol=1e-14, atol=0), 'Preserve direction across finite magnitude ranges'
    assert np.array_equal(gradient, before) and not np.shares_memory(clipped, gradient)
large = np.array([1e308, -1e308])
assert np.array_equal(clip_gradient(large, 1.5e308), large), 'Large representable norms below the limit must stay unchanged'
mixed = np.array([1e308, 1e-308])
assert np.allclose(clip_gradient(mixed, 5e307), [5e307, 5e-309], rtol=1e-14, atol=0), 'Retain small coordinates when their clipped values are representable'
empty = np.empty((0, 2))
assert clip_gradient(empty, 1).shape == empty.shape
assert np.allclose([learning_rate(i, 6, 2, 1, .1) for i in range(6)], [.5, 1, 1, .775, .325, .1])
assert learning_rate(0, 1, 0, 1, .2) == .2
assert learning_rate(1, 3, 2, 1e308) == 1e308, 'The final warmup update must reach the finite peak'
for peak in (1.,1e308,np.finfo(float).max):
    rates = np.array([learning_rate(step,5,4,peak) for step in range(4)])
    expected = peak*np.array([.25,.5,.75,1.])
    assert np.isfinite(rates).all() and np.allclose(rates,expected,rtol=1e-14,atol=0.)
for thunk in [lambda: loss_totals(np.zeros((1, 2, 3)), np.zeros((1, 2), int), np.zeros((1, 2), bool)), lambda: clip_gradient(g, 0), lambda: learning_rate(6, 6, 2, 1)]:
    try:
        thunk()
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid inputs must fail explicitly')
print('Training-accounting checks passed.')


## Existing Course Model

The exact vendored mini_llm configuration and model are embedded here so the notebook does not depend on a repository clone or network import. The Torch model includes learned normalization affine parameters and exact GELU; the browser capstone deliberately omits affine parameters and uses its documented approximation.


In [ ]:
from dataclasses import asdict, dataclass
from typing import Dict, Any


@dataclass
class ModelConfig:
    vocab_size: int
    block_size: int = 64
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64
    dropout: float = 0.1
    bias: bool = False

    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if self.vocab_size <= 0 or self.block_size <= 0:
            raise ValueError("vocab_size and block_size must be positive")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

"""A bigram baseline and a small decoder-only Transformer."""
import math
from typing import Dict, Optional, Tuple
import torch
import torch.nn as nn
from torch.nn import functional as F

def sample_next_token(logits: torch.Tensor, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
    if temperature <= 0:
        raise ValueError('temperature must be greater than zero')
    logits = logits / temperature
    if top_k is not None:
        top_k = min(top_k, logits.size(-1))
        threshold = torch.topk(logits, top_k).values[:, [-1]]
        logits = logits.masked_fill(logits < threshold, float('-inf'))
    probabilities = F.softmax(logits, dim=-1)
    return torch.multinomial(probabilities, num_samples=1)

class BigramLanguageModel(nn.Module):
    """Each token directly stores logits for the token that follows it."""

    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, vocab_size)
        nn.init.zeros_(self.token_embedding.weight)
        self.vocab_size = vocab_size

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        logits = self.token_embedding(inputs)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), targets.reshape(-1))
        return (logits, loss)

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            logits, _ = self(tokens[:, -1:])
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

class CausalSelfAttention(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head
        self.query_key_value = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.projection = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attention_dropout = nn.Dropout(config.dropout)
        self.residual_dropout = nn.Dropout(config.dropout)
        causal_mask = torch.tril(torch.ones(config.block_size, config.block_size))
        self.register_buffer('causal_mask', causal_mask.view(1, 1, config.block_size, config.block_size))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        batch_size, time_steps, channels = inputs.shape
        qkv = self.query_key_value(inputs)
        query, key, value = qkv.split(self.n_embd, dim=2)

        def split_heads(tensor: torch.Tensor) -> torch.Tensor:
            return tensor.view(batch_size, time_steps, self.n_head, self.head_dim).transpose(1, 2)
        query, key, value = map(split_heads, (query, key, value))
        attention = query @ key.transpose(-2, -1) / math.sqrt(self.head_dim)
        attention = attention.masked_fill(self.causal_mask[:, :, :time_steps, :time_steps] == 0, float('-inf'))
        weights = self.attention_dropout(F.softmax(attention, dim=-1))
        output = weights @ value
        output = output.transpose(1, 2).contiguous().view(batch_size, time_steps, channels)
        return self.residual_dropout(self.projection(output))

class FeedForward(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias), nn.GELU(), nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias), nn.Dropout(config.dropout))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.layers(inputs)

class TransformerBlock(nn.Module):

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.attention_norm = nn.LayerNorm(config.n_embd)
        self.attention = CausalSelfAttention(config)
        self.feed_forward_norm = nn.LayerNorm(config.n_embd)
        self.feed_forward = FeedForward(config)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        inputs = inputs + self.attention(self.attention_norm(inputs))
        inputs = inputs + self.feed_forward(self.feed_forward_norm(inputs))
        return inputs

class GPT(nn.Module):
    """A compact GPT-2-style decoder-only Transformer."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.final_norm = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, inputs: torch.Tensor, targets: Optional[torch.Tensor]=None) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        _, time_steps = inputs.shape
        if time_steps > self.config.block_size:
            raise ValueError(f'Sequence length {time_steps} exceeds block size {self.config.block_size}')
        positions = torch.arange(time_steps, device=inputs.device)
        hidden = self.token_embedding(inputs) + self.position_embedding(positions)
        hidden = self.dropout(hidden)
        for block in self.blocks:
            hidden = block(hidden)
        logits = self.lm_head(self.final_norm(hidden))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.config.vocab_size), targets.reshape(-1))
        return (logits, loss)

    def parameter_count(self, exclude_position_embeddings: bool=True) -> int:
        count = sum((parameter.numel() for parameter in self.parameters()))
        if exclude_position_embeddings:
            count -= self.position_embedding.weight.numel()
        return count

    def optimizer_groups(self, weight_decay: float) -> list:
        decay, no_decay = ([], [])
        for parameter in self.parameters():
            (decay if parameter.dim() >= 2 else no_decay).append(parameter)
        return [{'params': decay, 'weight_decay': weight_decay}, {'params': no_decay, 'weight_decay': 0.0}]

    @torch.no_grad()
    def generate(self, tokens: torch.Tensor, max_new_tokens: int, temperature: float=1.0, top_k: Optional[int]=None) -> torch.Tensor:
        for _ in range(max_new_tokens):
            context = tokens[:, -self.config.block_size:]
            logits, _ = self(context)
            next_token = sample_next_token(logits[:, -1, :], temperature, top_k)
            tokens = torch.cat((tokens, next_token), dim=1)
        return tokens

def build_model(model_type: str, config: ModelConfig) -> nn.Module:
    if model_type == 'bigram':
        return BigramLanguageModel(config.vocab_size)
    if model_type == 'gpt':
        return GPT(config)
    raise ValueError("model_type must be 'bigram' or 'gpt'")

def checkpoint_model(checkpoint: Dict[str, object], device: torch.device) -> nn.Module:
    raw_config = checkpoint.get('model_config')
    model_type = checkpoint.get('model_type')
    if not isinstance(raw_config, dict) or not isinstance(model_type, str):
        raise ValueError('Checkpoint is missing model metadata')
    model = build_model(model_type, ModelConfig(**raw_config))
    model.load_state_dict(checkpoint['model_state'])
    return model.to(device)


## Measured Experiment

Predict before running. Change one variable at a time. Report actual outcomes and failures; the lesson does not promise a successful learning curve for every configuration.


In [ ]:
"""CPU pretraining, token-weighted accumulation, and exact local resume.

Run: uv run --with numpy --with torch python experiments/scratch_training.py
The notebook embeds the existing mini_llm model source before this cell.
No downloaded data/weights. Fixed synthetic language, not natural-language skill.
"""
import copy
import hashlib
import io
import json
from pathlib import Path
import sys
import torch
from torch.nn import functional as F

if 'GPT' not in globals():
    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'colab/vendor/src'))
    from mini_llm.models import GPT
    from mini_llm.config import ModelConfig


def scratch_data(seed=19):
    """Independent unique sequences: next symbol usually advances by one."""
    generator = torch.Generator().manual_seed(seed)
    rows, seen = [], set()
    while len(rows) < 320:
        row = [int(torch.randint(8, (1,), generator=generator))]
        for _ in range(12):
            step = 1 if torch.rand((), generator=generator) < .82 else 3
            row.append((row[-1] + step) % 8)
        if tuple(row) not in seen:
            seen.add(tuple(row)); rows.append(row)
    data = torch.tensor(rows)
    permutation = torch.randperm(len(data), generator=generator)
    data = data[permutation]
    return data[:224], data[224:272], data[272:]


def scratch_pretrain(seed=7, updates=140):
    torch.set_num_threads(1)
    torch.manual_seed(seed)
    config = ModelConfig(vocab_size=8, block_size=12, n_layer=1, n_head=2, n_embd=24, dropout=0.)
    model = GPT(config)
    optimizer = torch.optim.AdamW(model.parameters(), lr=.006, weight_decay=.01)
    train, validation, test = scratch_data()
    generator = torch.Generator().manual_seed(seed + 1)

    def evaluate(rows):
        model.eval()
        with torch.no_grad():
            logits, _ = model(rows[:, :-1])
            return float(F.cross_entropy(logits.flatten(0, 1), rows[:, 1:].flatten()))

    def next_update(m, opt, gen):
        m.train()
        indices = torch.randint(len(train), (24,), generator=gen)
        batch = train[indices]
        opt.zero_grad(set_to_none=True)
        # Unequal microbatch sizes still implement the full token-average gradient.
        tokens = batch[:, 1:].numel()
        total_loss = 0.
        for micro in (batch[:7], batch[7:]):
            logits, _ = m(micro[:, :-1])
            loss_sum = F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum')
            loss = loss_sum / tokens
            loss.backward()
            total_loss += float(loss.detach())
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.)
        opt.step()
        return total_loss

    initial = evaluate(validation)
    history = []
    for step in range(updates):
        next_update(model, optimizer, generator)
        if (step + 1) % 35 == 0:
            history.append({'update': step + 1, 'train_nll': evaluate(train), 'validation_nll': evaluate(validation)})

    # Snapshot immediately before the next sampled update, including RNG state.
    checkpoint = copy.deepcopy(dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
        rng=generator.get_state(), torch_rng=torch.get_rng_state(), step=updates, cursor=0,
        config=config.to_dict(), tokenizer_id='symbols-a-through-h-v1'))
    serialized = io.BytesIO()
    torch.save(checkpoint, serialized)
    serialized.seek(0)
    checkpoint = torch.load(serialized, weights_only=True)

    continued = copy.deepcopy(model)
    continued_optimizer = torch.optim.AdamW(continued.parameters(), lr=.006, weight_decay=.01)
    continued_optimizer.load_state_dict(copy.deepcopy(optimizer.state_dict()))
    continued_rng = torch.Generator(); continued_rng.set_state(generator.get_state())
    with torch.random.fork_rng(devices=[]):
        restored = GPT(ModelConfig(**checkpoint['config']))
        restored.load_state_dict(checkpoint['model'])
        resumed_optimizer = torch.optim.AdamW(restored.parameters(), lr=.006, weight_decay=.01)
        resumed_optimizer.load_state_dict(checkpoint['optimizer'])
        resumed_rng = torch.Generator(); resumed_rng.set_state(checkpoint['rng'])
        torch.set_rng_state(checkpoint['torch_rng'])
        uninterrupted = next_update(continued, continued_optimizer, continued_rng)
        torch.set_rng_state(checkpoint['torch_rng'])
        resumed = next_update(restored, resumed_optimizer, resumed_rng)
    assert uninterrupted == resumed
    assert all(torch.equal(a, b) for a, b in zip(continued.parameters(), restored.parameters()))

    # Independent gradient-equivalence check before clipping/optimizer state.
    model.eval()
    batch = train[:9]
    optimizer.zero_grad(set_to_none=True)
    _, full_loss = model(batch[:, :-1], batch[:, 1:]); full_loss.backward()
    full_grads = [p.grad.clone() for p in model.parameters()]
    optimizer.zero_grad(set_to_none=True)
    for micro in (batch[:2], batch[2:]):
        logits, _ = model(micro[:, :-1])
        (F.cross_entropy(logits.flatten(0, 1), micro[:, 1:].flatten(), reduction='sum') / batch[:, 1:].numel()).backward()
    error = max(float((p.grad - g).abs().max()) for p, g in zip(model.parameters(), full_grads))
    assert error < 2e-6, error
    final = evaluate(validation)
    assert final < initial, 'This fixed toy run should learn its transition pattern'
    model.eval(); torch.manual_seed(seed + 100)
    sample = model.generate(torch.tensor([[0]]), 16, top_k=3).tolist()[0]
    report = dict(seed=seed, updates=updates, config=config.to_dict(), parameters=model.parameter_count(False),
        split_sizes=[len(train), len(validation), len(test)],
        data_sha256=hashlib.sha256(torch.cat([train, validation, test]).numpy().tobytes()).hexdigest(),
        initial_validation_nll=initial, final_validation_nll=final, test_nll=evaluate(test),
        accumulation_max_error=error, resume_exact=True, history=history, generated_ids=sample,
        tokenizer=dict(symbols=list('abcdefgh'),ids=list(range(8))),generated_text=''.join('abcdefgh'[i] for i in sample),
        limitation='Synthetic transition language; not a general-purpose LLM. Test used once after a fixed recipe.')
    return model, report


if __name__ == '__main__':
    _, report = scratch_pretrain()
    print(json.dumps(report, indent=2))
